In [23]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

In [24]:
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')
metaData = pd.read_csv('Data/metaData.csv')

train = pd.DataFrame(train)
test = pd.DataFrame(test)
metaData = pd.DataFrame(metaData)

In [25]:
train.shape

(221, 37)

In [26]:
train.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [27]:
feat = [col for col in train.columns if col not in ['event_id', 'time_to_hit_hours', 'event']]
X_train = train[feat].copy()
X_test = test[feat].copy()
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

In [28]:
event_col = train['event']
time_to_hit_col = train['time_to_hit_hours']

y_12 = ((event_col == 1) & (time_to_hit_col <= 12)).astype(int)
y_24 = ((event_col == 1) & (time_to_hit_col <= 24)).astype(int)
y_48 = ((event_col == 1) & (time_to_hit_col <= 48)).astype(int)
y_72 = ((event_col == 1) & (time_to_hit_col <= 72)).astype(int)

targets = {
    "12": y_12,
    "24": y_24,
    "48": y_48,
    "72": y_72
}

In [ ]:
#Train one classifier for each time horizon, then store the predicted probability of class 1 on the test set.
preds = {}
model = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=8,
        random_state=42,
        class_weight="balanced"
    )
for horizon, y in targets.items():
    model.fit(X_train, y)
    preds[horizon] = model.predict_proba(X_test)[:, 1]

In [32]:
p12 = preds["12"]
p24 = np.maximum(preds["24"], p12)
p48 = np.maximum(preds["48"], p24)
p72 = np.maximum(preds["72"], p48)

# Optional safety clip
p12 = np.clip(p12, 0, 1)
p24 = np.clip(p24, 0, 1)
p48 = np.clip(p48, 0, 1)
p72 = np.clip(p72, 0, 1)

In [36]:
submission = pd.DataFrame({
    "event_id": test["event_id"],
    "prob_12h": p12,
    "prob_24h": p24,
    "prob_48h": p48,
    "prob_72h": p72
})

In [38]:
submission.to_csv("submission.csv", index=False)
submission

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.177610,0.203233,0.203233,0.218505
1,13353600,0.513345,0.711474,0.711474,0.711474
2,13942327,0.112732,0.138789,0.170645,0.176791
3,16112781,0.615277,0.723229,0.735962,0.735962
4,17132808,0.702265,0.702265,0.702265,0.702265
...,...,...,...,...,...
90,94627327,0.373115,0.373115,0.382430,0.391134
91,96570675,0.144041,0.180521,0.180521,0.187566
92,97225766,0.148923,0.233341,0.233341,0.240522
93,98446281,0.315781,0.355002,0.355002,0.355002
